In [11]:
import numpy as np
import pandas as pd
import random
import time
import os
import csv
import logging
from datetime import datetime
from logging.handlers import RotatingFileHandler

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options # type: ignore
from selenium.common.exceptions import WebDriverException, NoSuchElementException, TimeoutException, ElementClickInterceptedException, StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [12]:
# logger mẫu
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def setup_driver(headless=False):
    chrome_options = Options()
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_argument("--start-maximized")
    chrome_options.add_argument("--disable-extensions")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option("useAutomationExtension", False)

    if headless:
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    return driver

In [13]:
BASE_FIELDNAMES = [
    "restaurant_id", "url", "restaurant_name", "restaurant_type", "rating_average",
    "num_of_reviews", "phone", "price_level", "address", "crawl_date"
]

In [14]:
# Hàm khởi tạo file CSV
def init_csv(output_file="restaurants_tripadvisor.csv"):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    if not os.path.isfile(output_file):
        with open(output_file, mode="w", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=BASE_FIELDNAMES)
            writer.writeheader()
        logger.info(f"Tạo file CSV mới: {output_file}")
    else:
        logger.info(f"File CSV {output_file} đã tồn tại.")


In [15]:
init_csv(
    r"D:\NCKH_ky2_25-26\CAPSTONE-PROJECT\Data\restaurants_tripadvisor.csv"
)

INFO:__main__:Tạo file CSV mới: D:\NCKH_ky2_25-26\CAPSTONE-PROJECT\Data\restaurants_tripadvisor.csv


In [ ]:
# Hàm click
def safe_click(driver, element, retries=3):
    for attempt in range(retries):
        try:
            WebDriverWait(driver, 5).until(
                lambda d: element.is_displayed() and element.is_enabled()
            )

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});", element
            )

            try:
                element.click()
            except Exception:
                driver.execute_script("arguments[0].click();", element)

            logger.info("Click thành công.")
            return True

        except StaleElementReferenceException:
            logger.warning(f"Element stale (lần {attempt+1}/{retries})")

        except Exception as e:
            logger.warning(f"Click thất bại (lần {attempt+1}/{retries}): {e}")

    logger.error("Click thất bại sau nhiều lần thử")
    return False


# 1. Lấy link + tên nhà hàng

In [16]:
if __name__ == "__main__":
    url = "http://tripadvisor.com/Restaurants-g298085-Da_Nang.html"

    driver = setup_driver(headless=False)
    driver.get(url)
    
    WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
    )

INFO:WDM:====== WebDriver manager ======
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Driver [C:\Users\ACER\.wdm\drivers\chromedriver\win64\143.0.7499.192\chromedriver-win32/chromedriver.exe] found in cache


In [18]:
def save_links(driver, output_file="restaurants_tripadvisor.csv"):
    restaurants = []

    # 1️⃣ Crawl dữ liệu từ TripAdvisor
    cards = driver.find_elements(
        By.CSS_SELECTOR,
        "div[data-automation='restaurantCard']"
    )

    for card in cards:
        try:
            link_container = card.find_element(
                By.CSS_SELECTOR,
                "div.XIWnB.z.y.M0"
            )
            a_tag = link_container.find_element(By.TAG_NAME, "a")

            url = a_tag.get_attribute("href")
            name = a_tag.text.strip()

            if url and name:
                restaurants.append({
                    "url": url,
                    "restaurant_name": name
                })

        except NoSuchElementException:
            continue

    # 2️⃣ Đọc CSV để lấy ID tiếp theo & tránh trùng
    existing_urls = set()
    next_id = 1

    if os.path.isfile(output_file):
        try:
            with open(output_file, mode="r", encoding="utf-8-sig") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if row.get("url"):
                        existing_urls.add(row["url"])
                    if row.get("restaurant_id"):
                        try:
                            next_id = max(next_id, int(row["restaurant_id"]) + 1)
                        except ValueError:
                            pass
        except Exception as e:
            logger.error(f"Lỗi khi đọc CSV: {e}")
            return

    # 3️⃣ Ghi dữ liệu mới vào CSV
    try:
        with open(output_file, mode="a", newline="", encoding="utf-8-sig") as f:
            writer = csv.DictWriter(f, fieldnames=BASE_FIELDNAMES)

            count_new = 0
            for r in restaurants:
                if r["url"] not in existing_urls:
                    writer.writerow({
                        "restaurant_id": next_id,
                        "url": r["url"],
                        "restaurant_name": r["restaurant_name"]
                    })
                    existing_urls.add(r["url"])
                    next_id += 1
                    count_new += 1

            logger.info(f"Đã thêm {count_new} nhà hàng mới vào CSV")

    except Exception as e:
        logger.error(f"Lỗi khi ghi CSV: {e}")

    return restaurants

In [19]:
save_links(
    driver,
    r"D:\NCKH_ky2_25-26\CAPSTONE-PROJECT\Data\restaurants_tripadvisor.csv"
)

INFO:__main__:Đã thêm 32 nhà hàng mới vào CSV


[{'url': 'https://www.tripadvisor.com/Restaurant_Review-g298085-d33234906-Reviews-Lua_Viet_Restaurant-Da_Nang.html',
  'restaurant_name': 'Lua Viet Restaurant'},
 {'url': 'https://www.tripadvisor.com/Restaurant_Review-g298085-d12254273-Reviews-Olivia_s_Prime_Steakhouse-Da_Nang.html',
  'restaurant_name': "Olivia's Prime Steakhouse"},
 {'url': 'https://www.tripadvisor.com/Restaurant_Review-g298085-d24982748-Reviews-An_Thoi_Da_Nang-Da_Nang.html',
  'restaurant_name': '1. An Thoi Da Nang'},
 {'url': 'https://www.tripadvisor.com/Restaurant_Review-g15296807-d13810289-Reviews-Thia_G_Da_N_ng-My_An_Da_Nang.html',
  'restaurant_name': '2. Thìa Gỗ Đà Nẵng'},
 {'url': 'https://www.tripadvisor.com/Restaurant_Review-g298085-d16891168-Reviews-M_C_Seafood_Restaurant-Da_Nang.html',
  'restaurant_name': '3. MỘC Seafood Restaurant'},
 {'url': 'https://www.tripadvisor.com/Restaurant_Review-g298085-d17760929-Reviews-B_p_Cu_n_Da_N_ng-Da_Nang.html',
  'restaurant_name': '4. Bếp Cuốn Đà Nẵng'},
 {'url': 'htt

# 2. Lấy thông tin chi tiết từng nhà hàng

In [20]:
if __name__ == "__main__":
    url = "https://www.tripadvisor.com.vn/Restaurant_Review-g298085-d24982748-Reviews-An_Thoi_Da_Nang-Da_Nang.html"

    driver = setup_driver(headless=False)
    driver.get(url)
    
    WebDriverWait(driver, 15).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)

INFO:WDM:====== WebDriver manager ======
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Get LATEST chromedriver version for google-chrome
INFO:WDM:Driver [C:\Users\ACER\.wdm\drivers\chromedriver\win64\143.0.7499.192\chromedriver-win32/chromedriver.exe] found in cache


In [21]:
def scrape_overview_info(driver, wait):
    data = {}

    try:
        data["rating_average"] = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "div[data-automation='bubbleRatingValue']")
            )
        ).text.strip()
    except Exception:
        data["rating_average"] = ""

    try:
        data["num_of_reviews"] = wait.until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "div[data-automation='bubbleReviewCount']")
            )
        ).text.strip()
    except Exception:
        data["num_of_reviews"] = ""

    try:
        data["price_level"] = wait.until(
            EC.visibility_of_element_located(
                (By.XPATH, "//span[contains(@class,'bTeln')]//span[contains(text(),'$')]")
            )
        ).text.strip()
    except Exception:
        data["price_level"] = ""

    try:
        data["address"] = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, "//span[contains(@data-automation,'MapLink')]")
            )
        ).text.strip()
    except Exception:
        data["address"] = ""

    try:
        phone_elem = wait.until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href^='tel:']"))
        )
        phone = phone_elem.text.strip()
        data["phone"] = phone or phone_elem.get_attribute("href").replace("tel:", "").strip()
    except Exception:
        data["phone"] = ""

    return data


In [22]:
def get_section_value(driver, title):
    try:
        section = driver.find_element(
            By.XPATH,
            f"""
            //div[
                contains(@class,'iPiKu')
                and .//div[
                    contains(@class,'csBwo')
                    and normalize-space(
                        translate(text(),'abcdefghijklmnopqrstuvwxyz','ABCDEFGHIJKLMNOPQRSTUVWXYZ')
                    )='{title.upper()}'
                ]
            ]
            """
        )

        # text value
        text_values = section.find_elements(
            By.XPATH, ".//div[contains(@class,'VImYz') and not(.//span)]"
        )
        if text_values:
            return ", ".join(v.text.strip() for v in text_values if v.text.strip())

        # list value (features)
        spans = section.find_elements(By.XPATH, ".//span")
        return ", ".join(s.text.strip() for s in spans if s.text.strip())

    except NoSuchElementException:
        return ""


In [ ]:
def scrape_restaurant_sections(driver):
    return {
        "cuisines": get_section_value(driver, "Cuisines"),
        "meal_types": get_section_value(driver, "Meal types"),
        "special_diets": get_section_value(driver, "Special Diets"),
        "features": get_section_value(driver, "Features"),
    }

In [25]:
def open_features_if_needed(driver, wait):
    # Nếu đã có section thì không cần click
    if driver.find_elements(By.XPATH, "//div[contains(@class,'csBwo')]"):
        return

    try:
        btn = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, "//button[.//span[normalize-space()='See all features']]")
            )
        )
        safe_click(driver, btn)
    except TimeoutException:
        logger.warning("Không tìm thấy nút See all features")


In [26]:
def scrape_restaurant_tripadvisor_detail(driver, wait):
    data = {
        **scrape_overview_info(driver, wait)
    }

    open_features_if_needed(driver, wait)

    data.update(
        scrape_restaurant_sections(driver)
    )

    return data


In [27]:
wait = WebDriverWait(driver, 10)

data = scrape_restaurant_tripadvisor_detail(driver, wait)

for key, value in data.items():
    print(f"{key}: {value}")

INFO:__main__:Click thành công.


rating_average: 4.9
num_of_reviews: (1,010 reviews)
price_level: $$ - $$$
address: 114 Bạch Đằng, Da Nang 550000 Vietnam
phone: +84 98 782 42 85
cuisines: Asian, Grill, Vietnamese, Healthy, Beer restaurants
meal_types: Breakfast, Lunch, Dinner
special_diets: Vegetarian friendly, Vegan options
features: Accepts Credit Cards, Delivery, Free Wifi, Highchairs Available, Mastercard, Seating, Serves Alcohol, Table Service, Takeout, Visa


In [ ]:
def update_details_and_save_tripadvisor(
    driver,
    output_file="restaurants_tripadvisor.csv",
    batch_size=10
):
    """
    Cập nhật thông tin chi tiết nhà hàng TripAdvisor
    (KHÔNG xử lý feature_type)

    Args:
        driver: Selenium WebDriver
        output_file: CSV chứa danh sách nhà hàng TripAdvisor
        batch_size: log tiến độ
    """
    wait = WebDriverWait(driver, 10)

    # 1️⃣ Đọc CSV
    try:
        with open(output_file, mode="r", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            if not all(col in reader.fieldnames for col in BASE_FIELDNAMES):
                logger.error(
                    f"CSV thiếu cột bắt buộc. Yêu cầu: {BASE_FIELDNAMES}"
                )
                return 0
            rows = list(reader)
    except FileNotFoundError:
        logger.error(f"Không tìm thấy file {output_file}")
        return 0
    except Exception as e:
        logger.error(f"Lỗi đọc CSV: {e}")
        return 0

    total = len(rows)
    updated = 0
    data_list = []

    # 2️⃣ Crawl từng nhà hàng
    for i, row in enumerate(rows):
        url = row.get("url", "")
        if not url:
            logger.warning(f"Dòng {i+1}: không có URL → bỏ qua")
            continue

        logger.info(
            f"Scraping {i+1}/{total}: {row.get('restaurant_name', 'Unknown')}"
        )

        try:
            driver.get(url)

            # Đợi title TripAdvisor
            wait.until(
                EC.presence_of_element_located(
                    (By.CSS_SELECTOR, "h1")
                )
            )

            # 3️⃣ Crawl chi tiết
            data = scrape_restaurant_tripadvisor(driver, wait)
            if not data:
                logger.warning(f"Không crawl được dữ liệu: {url}")
                continue

            has_change = False
            change_details = []

            current_data = {
                "restaurant_id": row.get("restaurant_id", ""),
                "url": url,
                "restaurant_name": row.get("restaurant_name", ""),
            }

            # 4️⃣ So sánh & cập nhật dữ liệu
            for key, new_val in data.items():
                old_val = row.get(key, "")

                new_val = str(new_val).strip()
                old_val = str(old_val).strip()

                if new_val == "" and old_val != "":
                    current_data[key] = old_val
                else:
                    current_data[key] = new_val
                    if new_val != old_val:
                        has_change = True
                        change_details.append(
                            f"{key}: '{old_val}' → '{new_val}'"
                        )

            # 5️⃣ crawl_date
            if has_change:
                current_data["crawl_date"] = datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                )
                updated += 1
                logger.info(
                    f"Cập nhật nhà hàng: {row.get('restaurant_name', url)}"
                )
                for d in change_details:
                    logger.info(f"  - {d}")
            else:
                current_data["crawl_date"] = row.get("crawl_date", "")
                logger.info(
                    f"Không thay đổi: {row.get('restaurant_name', url)}"
                )

            data_list.append(current_data)

        except (TimeoutException, NoSuchElementException) as e:
            logger.error(f"Lỗi scrape {url}: {e}")
            continue

        if (i + 1) % batch_size == 0:
            logger.info(f"Đã xử lý {i+1}/{total} nhà hàng")

    # 6️⃣ Ghi lại CSV
    if data_list:
        df = pd.DataFrame(data_list)
        df = df[BASE_FIELDNAMES]  # đảm bảo đúng thứ tự cột
        df.to_csv(
            output_file,
            index=False,
            encoding="utf-8-sig",
            quoting=csv.QUOTE_NONNUMERIC
        )
        logger.info(
            f"Hoàn thành! Đã cập nhật {updated} nhà hàng TripAdvisor"
        )
    else:
        logger.warning("Không có dữ liệu để ghi")

    return updated
